In [ ]:
import numpy as np
import torch
from medmnist import PathMNIST
from torchvision import transforms
from torch.utils.data import Subset
from tqdm import tqdm
import os

SEED = 42
np.random.seed(SEED)

os.makedirs("data/sample_224", exist_ok=True)

transform = transforms.ToTensor()

def criar_sample(split, quantidade, nome_arquivo):
    print(f"Carregando split {split}...")

    dataset = PathMNIST(
        split=split,
        size=224,
        transform=transform,
        download=True
    )

    total = len(dataset)
    indices = np.random.choice(total, size=quantidade, replace=False)

    imagens = []
    labels = []

    print(f"Criando sample de {quantidade} imagens do split {split}...")

    for idx in tqdm(indices):
        img, label = dataset[idx]

        # Salva como uint8 para ocupar menos espaço
        img_uint8 = (img.numpy() * 255).astype(np.uint8)

        imagens.append(img_uint8)
        labels.append(label)

    imagens = np.stack(imagens)
    labels = np.array(labels)

    caminho = f"data/sample_224/{nome_arquivo}"

    np.savez_compressed(
        caminho,
        images=imagens,
        labels=labels,
        indices=indices
    )

    print(f"Arquivo salvo em: {caminho}")
    print("Formato imagens:", imagens.shape)
    print("Formato labels:", labels.shape)


criar_sample("train", 3000, "train_sample_224.npz")
criar_sample("val", 800, "val_sample_224.npz")
criar_sample("test", 1000, "test_sample_224.npz")

In [1]:
from pathlib import Path
import numpy as np

print("Diretório atual:", Path.cwd())

caminhos_possiveis = [
    Path("data/sample_224"),
    Path("notebooks/data/sample_224"),
]

base_path = None

for caminho in caminhos_possiveis:
    if caminho.exists():
        base_path = caminho
        break

if base_path is None:
    raise FileNotFoundError(
        "A pasta data/sample_224 não foi encontrada. "
        "Confira onde os arquivos .npz estão salvos."
    )

print("Pasta encontrada:", base_path.resolve())

arquivos = {
    "train_sample_224.npz": 3000,
    "val_sample_224.npz": 800,
    "test_sample_224.npz": 1000,
}

for nome, quantidade_esperada in arquivos.items():
    caminho = base_path / nome

    if not caminho.exists():
        print(f"❌ Arquivo não encontrado: {caminho}")
        continue

    with np.load(caminho) as dados:
        shape_imagens = dados["images"].shape
        shape_labels = dados["labels"].shape

    print(f"\n✅ {nome}")
    print("  Imagens:", shape_imagens)
    print("  Labels:", shape_labels)

    formatos_validos = [
    (quantidade_esperada, 224, 224, 3),  # HWC
    (quantidade_esperada, 3, 224, 224),  # CHW
]

assert shape_imagens in formatos_validos, (
    f"Formato inesperado em {nome}: {shape_imagens}"
)

assert shape_labels in [
    (quantidade_esperada,),
    (quantidade_esperada, 1),
], f"Formato inesperado nos labels de {nome}: {shape_labels}"

Diretório atual: c:\Users\rafae\OneDrive\Desktop\trabalho-final-ia\notebooks
Pasta encontrada: C:\Users\rafae\OneDrive\Desktop\trabalho-final-ia\notebooks\data\sample_224

✅ train_sample_224.npz
  Imagens: (3000, 3, 224, 224)
  Labels: (3000, 1)

✅ val_sample_224.npz
  Imagens: (800, 3, 224, 224)
  Labels: (800, 1)

✅ test_sample_224.npz
  Imagens: (1000, 3, 224, 224)
  Labels: (1000, 1)
